# UKB SER-fallback cases: Q_art and D_CS (data processing)

From `34.UKB_SER_Fallback_Cases` (`2026_output/UKB_SER_fallback_cases_1000G/`, or `UKB_SER_fallback_cases/` with `ld_ref = "ADSP"`): UKB blood-trait region × trait pairs (1000 Genomes EUR LD; or the ADSP sketch) where SuSiE-rRSS falls back to the SER model, by trigger: **Q_art only**, **D_CS only**, **both**. Candidates were drawn from the stored coloc diagnostics (SuSiE-RSS with ≥ 2 credible sets, Q_art ≥ 0.15 / D_CS,max ≥ 60, largest regions first) and refitted with the current susieR. A case is kept when the refit has the same trigger and still meets these criteria, in candidate order, up to 8 / 8 / 5.

- **Q_art** = ‖r_low‖² / ‖r‖²: r is the z-scale fitted residual of the SuSiE-rRSS fit, r_low its projection on the eigenvectors of the reference LD with eigenvalue ≤ δ (= 10⁻³ × the largest). The numerator and denominator are saved so it is visible which one drives the ratio.
- **D_CS** of a credible set = max over its variants of (nominal − adjusted) log BF of its effect, i.e. the log of the ratio of the BF with uninflated s.e. to the BF with s.e. inflated for R uncertainty. D_CS,max is the largest over the credible sets of the fit before fallback. The adjusted and nominal log BF of the top variant are saved.

**Outputs:**
- `figure_data/UKB_SER_Fallback_Cases_1000G.rds` (no csv, too large): one row per unit × variant: position, z, −log10 p, lead; PIP / credible set (`0` = none) of in-sample SuSiE-RSS, SuSiE-RSS, SuSiE-rRSS before fallback (`rrss_full`) and the SuSiE-rRSS result (`rrss`: SER when flagged); `r_fit`, `r_low` (Q_art); adjusted / nominal log BF and attenuation of the effect behind the top-D_CS credible set (`lbf_adj`, `lbf_nom`, `atten`, `top_cs`); per-variant s.e.² inflation of the last SER update.
- `figure_data/UKB_SER_Fallback_Cases_1000G_Summary.rds` (+ `.csv`): one row per unit: Q_art with numerator / denominator, D_CS,max with its CS, top variant and log BF, flags, trigger (stored and refit), credible-set counts, `shown`.
- `figure_data/UKB_SER_Fallback_Cases_1000G_CS.rds` (+ `.csv`): one row per credible set of SuSiE-rRSS before fallback: D_CS, top variant, its adjusted / nominal log BF.
- `figure_data/UKB_SER_Fallback_Cases_1000G_Eigen.rds`: one row per unit × eigen-direction of the reference LD: eigenvalue relative to the largest, and the share of ‖r‖² along it (shares with eigenvalue ≤ δ sum to Q_art).

In [4]:
source("../../R/utils.R")

ld_ref    <- "1000G"     # "1000G" (UKB_SER_fallback_cases_1000G) or "ADSP" (UKB_SER_fallback_cases)
sfx       <- if (ld_ref == "1000G") "_1000G" else ""
data_name <- paste0("UKB_SER_Fallback_Cases", sfx)
base      <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_SER_fallback_cases", sfx)
root      <- file.path(base, "result")
n_show    <- c("Q_art only" = 8, "D_CS only" = 8, "Both" = 5)
q_min     <- 0.15; dcs_min <- 60; min_rss_cs <- 2      # as 34 [fallback_cases_select]
trig_lev  <- names(n_show)

cases <- read_tsv(file.path(base, "units", "fallback_cases.tsv"), show_col_types = FALSE) %>%
  filter(candidate) %>% mutate(trigger = factor(trigger, levels = trig_lev))
done  <- function(u) all(file.exists(file.path(root, u, paste0(u, ".", c("inputs", "susie_rss", "susie_rrss", "insample"), ".rds"))))
units <- cases$unit[map_lgl(cases$unit, done)]
cat(length(units), "of", nrow(cases), "candidates fitted\n")
setdiff(cases$unit, units)

14 of 14 candidates fitted


character(0)

In [5]:
## credible-set index per variant (0 = none)
cs_index <- function(fit, p) {
  cs <- rep(0L, p)
  for (k in seq_along(fit$sets$cs)) cs[fit$sets$cs[[k]]] <- k
  cs
}

one_unit <- function(u) {
  f   <- function(m) file.path(root, u, paste0(u, ".", m, ".rds"))
  inp <- readRDS(f("inputs"))
  p   <- length(inp$z_sketch)
  lead <- which.max(abs(inp$z_sketch))
  rss <- readRDS(f("susie_rss")); rrss <- readRDS(f("susie_rrss")); ins <- readRDS(f("insample"))
  for (o in list(rss, rrss, ins)) if (!is.null(o$error)) stop(u, " ", o$method, ": ", o$error)
  full <- rrss$full_fit
  stopifnot(length(full$pip) == p, !is.null(full$keep_R_bf_attenuation), !is.null(rrss$qart))
  if (!isTRUE(rrss$qart_matches_fit)) warning(u, ": saved Q_art residual does not match the fit's Q_art")

  ## D_CS: per credible set of the fit before fallback, max over its variants of
  ## (nominal - adjusted) log BF of its effect l; the top CS drives D_CS,max
  A   <- full$keep_R_bf_attenuation; LBF <- full$lbf_variable
  csl <- full$sets$cs_index %||% as.integer(sub("^L", "", names(full$sets$cs)))
  cs_tab <- map_dfr(seq_along(full$sets$cs), function(i) {
    l <- csl[i]; v <- full$sets$cs[[i]]; a <- A[l, v]; j <- v[which.max(replace(a, !is.finite(a), -Inf))]
    tibble(unit = u, cs = i, effect = l, size = length(v), D_CS = max(a, na.rm = TRUE), top_var = j,
           top_alpha = full$alpha[l, j], lbf_adj = LBF[l, j], lbf_nom = LBF[l, j] + A[l, j],
           inflation = full$keep_shat2_inflation[j])
  })
  top <- if (nrow(cs_tab)) slice_max(cs_tab, D_CS, n = 1, with_ties = FALSE) else NULL
  l_top <- if (!is.null(top)) top$effect else NA_integer_

  q <- rrss$qart
  var <- tibble(unit = u, idx = seq_len(p), variant = inp$variant_ids, pos = inp$sumstat$pos, z = inp$z_sketch,
                logp = -(pnorm(-abs(inp$z_sketch), log.p = TRUE) + log(2)) / log(10),
                lead = seq_len(p) == lead,
                r_fit = q$r_fit, r_low = q$r_low,                            # Q_art = sum(r_low^2) / sum(r_fit^2)
                inflation = full$keep_shat2_inflation,
                ## effect of the top-D_CS credible set: adjusted and nominal log BF
                lbf_adj = if (is.na(l_top)) NA_real_ else LBF[l_top, ],
                atten   = if (is.na(l_top)) NA_real_ else A[l_top, ],
                lbf_nom = lbf_adj + pmax(atten, 0))
  fits <- list(insample = ins$fit, rss = rss$fit, rrss_full = full, rrss = rrss$fit)  # rrss: pipeline result (SER when flagged)
  for (m in names(fits)) {
    var[[paste0("pip_", m)]] <- as.numeric(fits[[m]]$pip)
    var[[paste0("cs_", m)]]  <- cs_index(fits[[m]], p)
  }
  var$top_cs <- if (is.na(l_top)) FALSE else var$cs_rrss_full == top$cs

  q_art <- rrss$Q_art; dcs <- if (nrow(cs_tab)) max(cs_tab$D_CS) else NA_real_
  summ <- tibble(unit = u, region = inp$region, trait = inp$trait, p = p, n = inp$n, B = inp$B_sketch,
                 lead_idx = lead, max_logp = max(var$logp),
                 Q_art = q_art, Q_num = sum(q$r_low^2), Q_den = sum(q$r_fit^2),
                 low_eigen = q$low_eigen_count, eig_delta = q$eig_delta, Q_evaluable = q$evaluable,
                 D_CS_max = dcs, top_cs = top$cs %||% NA_integer_, top_var = top$top_var %||% NA_integer_,
                 top_lbf_adj = top$lbf_adj %||% NA_real_, top_lbf_nom = top$lbf_nom %||% NA_real_,
                 top_inflation = top$inflation %||% NA_real_, top_cs_size = top$size %||% NA_integer_,
                 artifact_flag = rrss$artifact_flag, sensitivity_flag = rrss$sensitivity_flag,
                 ser_fallback = rrss$ser_fallback,
                 trigger_refit = case_when(rrss$artifact_flag & rrss$sensitivity_flag ~ "Both",
                                           rrss$artifact_flag ~ "Q_art only",
                                           rrss$sensitivity_flag ~ "D_CS only", TRUE ~ "none"),
                 n_cs_insample = length(ins$fit$sets$cs), n_cs_rss = length(rss$fit$sets$cs),
                 n_cs_rrss_full = length(full$sets$cs), n_cs_rrss = length(rrss$fit$sets$cs))
  ## residual along every eigen-direction of the reference LD (shares with eigenvalue <= delta sum to Q_art)
  eig <- tibble(unit = u, lambda_rel = q$eig_values / q$eig_max, share = q$eig_proj2 / sum(q$r_fit^2),
                low = q$eig_values <= q$eig_delta)
  stopifnot(isTRUE(all.equal(sum(eig$share[eig$low]), sum(q$r_low^2) / sum(q$r_fit^2))))
  list(var = var, summ = summ, cs = cs_tab, eig = eig)
}

res  <- map(units, safely(one_unit))
errs <- map(res, "error"); names(errs) <- units
compact(errs)
res  <- compact(map(res, "result"))
var    <- map_dfr(res, "var")
summ   <- map_dfr(res, "summ")
cs_tab <- map_dfr(res, "cs")
eig    <- map_dfr(res, "eig")

named list()

In [6]:
## keep the cases whose refit reproduces the stored trigger and still meets the selection criteria,
## in candidate order, up to n_show per class
summ <- summ %>%
  left_join(select(cases, unit, trigger, rank, Q_art_stored = Q_art, DCS_max_stored = DCS_max), by = "unit") %>%
  mutate(reproduced = trigger_refit == as.character(trigger) & n_cs_rss >= min_rss_cs &
                      (trigger == "D_CS only"  | Q_art >= q_min) &
                      (trigger == "Q_art only" | D_CS_max >= dcs_min)) %>%
  group_by(trigger) %>% arrange(!reproduced, rank, .by_group = TRUE) %>%
  mutate(shown = reproduced & cumsum(reproduced) <= n_show[as.character(trigger[1])]) %>%
  ungroup()

summ %>% count(trigger, reproduced, shown) %>% as.data.frame()
summ %>% select(trigger, trigger_refit, rank, shown, unit, Q_art_stored, Q_art, Q_num, Q_den,
                DCS_max_stored, D_CS_max, top_lbf_nom, top_lbf_adj, n_cs_rss, n_cs_rrss_full, n_cs_rrss) %>%
  mutate(across(where(is.double), ~ signif(.x, 3))) %>% as.data.frame()

save_data(var,    data_name, csv = FALSE)                                 # ~40 MB as csv: rds only
save_data(summ,   paste0(data_name, "_Summary"))
save_data(cs_tab, paste0(data_name, "_CS"))
save_data(eig,    paste0(data_name, "_Eigen"), csv = FALSE)

trigger,reproduced,shown,n
<fct>,<lgl>,<lgl>,<int>
D_CS only,TRUE,FALSE,4
D_CS only,TRUE,TRUE,8
Both,TRUE,TRUE,2


trigger,trigger_refit,rank,shown,unit,Q_art_stored,Q_art,Q_num,Q_den,DCS_max_stored,D_CS_max,top_lbf_nom,top_lbf_adj,n_cs_rss,n_cs_rrss_full,n_cs_rrss
<fct>,<chr>,<dbl>,<lgl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<int>,<int>
D_CS only,D_CS only,1,TRUE,chr12_108470446_113450293.Platelet_count,0.0489,0.0489,2150,43900,4560.0,4560.0,6090,1530,9,8,1
D_CS only,D_CS only,2,TRUE,chr7_97615936_102353188.RBC_count,0.0508,0.0508,1400,27700,80.1,80.1,310,229,7,4,1
D_CS only,D_CS only,3,TRUE,chr11_63608227_67817936.RDW,0.0346,0.0346,999,28800,11200.0,11200.0,13400,2160,7,5,1
D_CS only,D_CS only,4,TRUE,chr1_91337561_95126928.Monocyte_perc,0.0351,0.0351,1170,33400,92.8,92.8,541,448,6,2,1
D_CS only,D_CS only,5,TRUE,chr16_66168241_69809315.Eosinophill_perc,0.0373,0.0373,594,15900,390.0,390.0,1300,908,3,3,0
D_CS only,D_CS only,6,TRUE,chr8_40707809_44076321.MSCV,0.0567,0.0567,1700,30000,356.0,356.0,789,433,9,5,1
D_CS only,D_CS only,7,TRUE,chr1_170579505_173481898.PDW,0.0326,0.0326,971,29800,689.0,689.0,866,176,9,3,1
D_CS only,D_CS only,8,TRUE,chr9_4427630_7307909.Platelet_count,0.0557,0.0557,1540,27600,201.0,201.0,372,171,9,7,1
D_CS only,D_CS only,9,FALSE,chr1_39041708_41794279.RDW,0.0435,0.0435,562,12900,728.0,728.0,1820,1100,5,5,1
